In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# IDs to highlight (specified after removing '-P')
EMPH_IDS = {
    "AO090003001208",
    "AO090012000267",
    "AO090003001292",
    "AO090012000035",
    "AO090038000235",
    "AO090113000137",
    "AO090701000449",
    "AO090026000014",
    "AO090026000614",
    "AO090012000182",
    "AO090012000002_1",
    "AO090012000002_2",
    "AO090005001236",  # focal case near a beta-lactone BGC
}

# Forty-eight IDs near BGCs (copied from Near_BGC_AoID_list)
# Display these as triangles rather than circles in the PCA plots.
NEAR_BGC_IDS = set(['AO090009000717', 'AO090009000079', 'AO090009000133', 'AO090005001545', 'AO090005001236', 'AO090005001228', 'AO090005001087', 'AO090005001663', 'AO090005000990', 'AO090005000933', 'AO090001000753', 'AO090001000029', 'AO090001000755', 'AO090001000756', 'AO090001000372', 'AO090001000392', 'AO090001000515', 'AO090003000962', 'AO090003001543', 'AO090023000280', 'AO090023000305', 'AO090023000416', 'AO090026000014', 'AO090012000525', 'AO090102000477', 'AO090102000449', 'AO090701000776', 'AO090701000217', 'AO090113000035', 'AO090113000164', 'AO090113000204', 'AO090020000384', 'AO090020000136', 'AO090038000557', 'AO090038000532', 'AO090038000494', 'AO090038000085', 'AO090038000032', 'AO090011000961', 'AO090011000729', 'AO090103000377', 'AO090010000442', 'AO090010000338', 'AO090010000784', 'AO090010000324', 'AO090010000113', 'AO090010000049', 'AO090010000037'])
assert len(NEAR_BGC_IDS) == 48, f"The number of near-BGC IDs is not 48: {len(NEAR_BGC_IDS)}"

# Default colors for support-prioritized display groups (fallback when no text file is supplied)
DEFAULT_GROUP_COLORS = {
    "G1": "#1f77b4",
    "G2": "#ff7f0e",
    "G3": "#2ca02c",
    "G4": "#d62728",
    "G5": "#9467bd",
    "G6": "#8c564b",
    "G7": "#e377c2",
    "G8": "#7f7f7f",
    "G9": "#bcbd22",
    "G10": "#17becf",
}


# Features protected during correlation filtering
# For this analysis, no features are protected; for each correlation component with |r| > corr_threshold,
# retain only one representative feature.
PROTECTED_MOLEPCT_FEATURES = []


In [ ]:
# -----------------------------
# ID preprocessing (remove text after |, add duplicate suffixes, and remove "-P")
# -----------------------------
def clean_id_series(id_series, mode):
    """
    mode:
      - "SM00066": aaComp.csv input (Domain; append _1, _2, ... to duplicate IDs)
      - "FL":      FL_aaComp.csv input
      - "non_DBD": non_DBD_aaComp.csv input
    """

    s = id_series.astype(str)

    # Remove "|" and everything after it
    base = s.str.split("|", n=1, expand=True)[0]

    # Remove "-P"
    base = base.str.replace("-P", "", regex=False)

    if mode == "SM00066":

        # Number of duplicates
        counts = base.groupby(base).transform("count")

        # Order of occurrence
        dup_index = base.groupby(base).cumcount()

        # Assign _1, _2, ... to every duplicated ID
        base = np.where(
            counts > 1,
            base + "_" + (dup_index + 1).astype(str),
            base
        )

        return pd.Series(base, index=id_series.index)

    elif mode in {"FL", "non_DBD"}:
        return base

    else:
        raise ValueError("mode must be 'SM00066', 'FL', or 'non_DBD'")


In [ ]:
# -----------------------------
# Read the iTOL color strip (support-prioritized display groups)
# -----------------------------
def load_itol_colorstrip(txt_path):
    """
    Returns
    -------
    id_to_color : dict
        sample ID -> hex color
    id_to_group : dict
        sample ID -> display-group label (G1--G10)
    group_to_color : dict
        display-group label -> hex color

    Notes
    -----
    Use G1--G10 from the current file. If an older file is loaded accidentally,
    normalize C1--C10 to G1--G10. These are not biological clade names;
    they are support-prioritized display groups.
    """
    import re

    def normalize_group_label(label):
        label = str(label).strip()
        m = re.match(r"^[CG](\d+)(?:\s|$)", label)
        return f"G{int(m.group(1))}" if m else label

    id_to_color = {}
    id_to_group = {}
    group_to_color = {}
    legend_colors = []

    in_data = False
    with open(txt_path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if (not line) or line.startswith("#"):
                continue

            if line == "DATA":
                in_data = True
                continue

            parts = line.split("\t")

            if not in_data:
                if parts[0] == "LEGEND_COLORS":
                    legend_colors = parts[1:]
                elif parts[0] == "LEGEND_LABELS" and legend_colors:
                    for lab, col in zip(parts[1:], legend_colors):
                        group_to_color[normalize_group_label(lab)] = str(col)
                continue

            if len(parts) >= 3:
                sid, color, raw_group = parts[0], parts[1], parts[2]
                group = normalize_group_label(raw_group)
                id_to_color[sid] = color
                id_to_group[sid] = group
                group_to_color.setdefault(group, color)

    if not group_to_color:
        group_to_color = DEFAULT_GROUP_COLORS.copy()

    observed = set(id_to_group.values())
    unexpected = sorted(observed - set(DEFAULT_GROUP_COLORS))
    if unexpected:
        print("Warning: unexpected display-group labels:", unexpected)

    return id_to_color, id_to_group, group_to_color

# Note:
# CLR transformation is intentionally not used in this StandardScaler-only version.
# The MolePct columns used here are overlapping physicochemical property categories,
# not mutually exclusive compositional parts that sum to 100%.


In [ ]:
# -----------------------------
# Remove highly correlated features (clustering method: Method B; no protected features)
#   - Treat |r| > threshold as belonging to the same component and retain at least one representative feature per component
#   - Always retain features listed in protected_features, even within a correlated component
#   - When a component contains no protected feature, select the feature with the largest variance by default
# Returns: (X_reduced, dropped_cols, corr_full, cluster_df)
# -----------------------------
def drop_highly_correlated_features(
    X,
    threshold=0.60,
    representative="variance",
    protected_features=None
):
    """
    Method B without protected features in the present analysis:
      1) Compute correlation on X
      2) Build graph where an edge connects features with |r| > threshold
      3) Find connected components (clusters)
      4) Keep one representative feature per cluster unless protected_features is explicitly provided

    Parameters
    ----------
    X : pandas.DataFrame
        Feature matrix (rows=samples, cols=features)
    threshold : float
        Absolute correlation cutoff for clustering
    representative : str
        How to choose cluster representative when no protected feature exists:
          - "variance": keep feature with largest variance in X within the cluster
          - "mean_abs_corr": keep feature with smallest mean(|r|) to others in the cluster
    protected_features : list[str] or None
        Features that must always be retained if present in X

    Returns
    -------
    X_reduced : DataFrame
    dropped_cols : list[str]
    corr_full : DataFrame
    cluster_df : DataFrame
        feature -> cluster_id, kept/dropped, protection status, representative status
    """
    import numpy as np
    import pandas as pd

    corr_full = X.corr()
    cols = list(corr_full.columns)

    if protected_features is None:
        protected_features = []
    protected_features = [str(f) for f in protected_features if str(f) in cols]
    protected_set = set(protected_features)

    # Build adjacency list for graph of high-|r| edges
    abs_corr = corr_full.abs()
    adj = {c: set() for c in cols}
    for i in range(len(cols)):
        ci = cols[i]
        for j in range(i + 1, len(cols)):
            cj = cols[j]
            if abs_corr.iloc[i, j] > threshold:
                adj[ci].add(cj)
                adj[cj].add(ci)

    # Connected components (DFS)
    visited = set()
    clusters = []
    for c in cols:
        if c in visited:
            continue
        stack = [c]
        comp = []
        visited.add(c)
        while stack:
            v = stack.pop()
            comp.append(v)
            for nb in adj[v]:
                if nb not in visited:
                    visited.add(nb)
                    stack.append(nb)
        clusters.append(sorted(comp))

    def _choose_representative(comp):
        if len(comp) == 1:
            return comp[0]
        if representative == "mean_abs_corr":
            sub = abs_corr.loc[comp, comp].copy()
            np.fill_diagonal(sub.values, np.nan)
            mean_abs = sub.mean(axis=1, skipna=True)
            return mean_abs.idxmin()
        return X_var.loc[comp].idxmax()

    kept = []
    dropped = []
    rows = []

    X_var = X.var(axis=0)  # variance per feature

    for cid, comp in enumerate(clusters, start=1):
        protected_in_comp = [f for f in comp if f in protected_set]
        representative_feature = _choose_representative(comp)

        # Keep all protected features; otherwise keep one representative
        if protected_in_comp:
            keep_in_comp = list(dict.fromkeys(protected_in_comp))
        else:
            keep_in_comp = [representative_feature]

        kept.extend(keep_in_comp)

        for f in comp:
            is_kept = f in keep_in_comp
            if not is_kept:
                dropped.append(f)

            if f in protected_set:
                kept_reason = "protected"
            elif (not protected_in_comp) and (f == representative_feature):
                kept_reason = "representative"
            else:
                kept_reason = "dropped_by_correlation"

            rows.append({
                "feature": f,
                "cluster_id": cid,
                "is_protected": f in protected_set,
                "is_representative": f == representative_feature,
                "kept": is_kept,
                "dropped": not is_kept,
                "kept_reason": kept_reason,
                "cluster_size": len(comp)
            })

    kept = list(dict.fromkeys([c for c in cols if c in set(kept)]))  # preserve original column order
    dropped_cols = sorted(set(dropped))

    X_reduced = X[kept].copy()

    cluster_df = (
        pd.DataFrame(rows)
        .sort_values(["cluster_id", "dropped", "is_protected", "feature"],
                     ascending=[True, True, False, True])
        .reset_index(drop=True)
    )

    return X_reduced, dropped_cols, corr_full, cluster_df


In [ ]:
# -----------------------------
# Correlation matrix export (CSV + heatmap PNG/PDF)
#   - Keeps file naming consistent across aaComp/scale pipelines
#   - Optionally exports cluster membership (Method B)
# -----------------------------
def export_corr_outputs(corr_full, corr_reduced, dropped_cols, out_prefix, dpi=300, cluster_df=None):
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    # CSVs
    pd.DataFrame(corr_full).to_csv(f"{out_prefix}_corr_full.csv")
    pd.DataFrame(corr_reduced).to_csv(f"{out_prefix}_corr_reduced.csv")
    pd.Series(dropped_cols, name="DroppedFeatures").to_csv(f"{out_prefix}_dropped_features.csv", index=False)

    if cluster_df is not None:
        pd.DataFrame(cluster_df).to_csv(f"{out_prefix}_corr_clusters.csv", index=False)

    # Heatmaps (PNG + PDF)
    for corr, tag in [(corr_full, "corr_full"), (corr_reduced, "corr_reduced")]:
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(pd.DataFrame(corr), cmap="RdBu_r", center=0, ax=ax)
        ax.set_title(f"{out_prefix}: {tag}")
        fig.tight_layout()
        fig.savefig(f"{out_prefix}_{tag}.png", dpi=dpi)
        fig.savefig(f"{out_prefix}_{tag}.pdf")
        plt.close(fig)

    print("Saved:")
    print(f"  - {out_prefix}_corr_full.csv")
    print(f"  - {out_prefix}_corr_reduced.csv")
    print(f"  - {out_prefix}_dropped_features.csv")
    if cluster_df is not None:
        print(f"  - {out_prefix}_corr_clusters.csv")
    print(f"  - {out_prefix}_corr_full.png/.pdf")
    print(f"  - {out_prefix}_corr_reduced.png/.pdf")


In [ ]:
# -----------------------------
# Biplot（phylogenetic display-group colors, focal-ID highlighting, and triangles for near-BGC genes）
#   - Standard points: circles colored by display group
#   - Forty-eight near-BGC IDs: triangles colored by display group
#   - Previously highlighted IDs: black outline, larger size, and ID labels (near-BGC status determines shape)
#   - feature vectors + labels: red
#   - Retain suffix-insensitive matching
# -----------------------------
def pca_biplot_emph_only(
    scores_2d,
    loadings_2d,
    feature_names,
    sample_ids,
    title,
    out_prefix,
    dpi=600,
    vector_length_multiplier=2.0,
    emph_ids=None,
    near_bgc_ids=None,
    id_to_color=None,
    id_to_group=None,
    group_to_color=None,
    default_point_color="#bdbdbd"
):
    import numpy as np
    import re
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    if emph_ids is None:
        emph_ids = EMPH_IDS
    if near_bgc_ids is None:
        near_bgc_ids = NEAR_BGC_IDS
    if id_to_color is None:
        id_to_color = {}
    if id_to_group is None:
        id_to_group = {}
    if group_to_color is None:
        group_to_color = DEFAULT_GROUP_COLORS.copy()

    sample_ids = np.array(list(map(str, sample_ids)))

    def _base_id(s):
        return re.sub(r"_\d+$", "", str(s))

    def _lookup(mapping, sid):
        if sid in mapping:
            return mapping[sid]
        sid_base = _base_id(sid)
        if sid_base in mapping:
            return mapping[sid_base]
        # Handle cases where only one side has a suffix
        suffix_hits = [v for k, v in mapping.items() if _base_id(k) == sid_base]
        if len(suffix_hits) > 0:
            return suffix_hits[0]
        return None

    def _membership_mask(ids):
        id_set = set(map(str, ids))
        base_set = {_base_id(s) for s in id_set}
        return np.array([
            (sid in id_set) or (_base_id(sid) in base_set)
            for sid in sample_ids
        ])

    is_emph = _membership_mask(emph_ids)
    is_near_bgc = _membership_mask(near_bgc_ids)

    point_colors = np.array([
        _lookup(id_to_color, sid) if _lookup(id_to_color, sid) is not None else default_point_color
        for sid in sample_ids
    ], dtype=object)

    point_groups = np.array([
        _lookup(id_to_group, sid) if _lookup(id_to_group, sid) is not None else "Unassigned"
        for sid in sample_ids
    ], dtype=object)

    fig, ax = plt.subplots(figsize=(8.8, 7.4))

    # Standard IDs: circles (excluding previously highlighted IDs)
    circle_normal = (~is_near_bgc) & (~is_emph)
    ax.scatter(
        scores_2d[circle_normal, 0], scores_2d[circle_normal, 1],
        s=22, c=point_colors[circle_normal], marker="o", alpha=0.9, zorder=1
    )

    # Near-BGC IDs: triangles. Retain display-group colors and use black outlines for clarity.
    triangle_normal = is_near_bgc & (~is_emph)
    if triangle_normal.any():
        ax.scatter(
            scores_2d[triangle_normal, 0], scores_2d[triangle_normal, 1],
            s=58, c=point_colors[triangle_normal], marker="^",
            edgecolors="black", linewidths=0.65, alpha=0.95, zorder=2
        )

    # Previously highlighted IDs: large circles with black outlines
    circle_emph = is_emph & (~is_near_bgc)
    if circle_emph.any():
        ax.scatter(
            scores_2d[circle_emph, 0], scores_2d[circle_emph, 1],
            s=86, c=point_colors[circle_emph], marker="o", edgecolors="black",
            linewidths=1.0, zorder=3
        )

    # IDs that are both near a BGC and previously highlighted: large triangles with black outlines
    triangle_emph = is_near_bgc & is_emph
    if triangle_emph.any():
        ax.scatter(
            scores_2d[triangle_emph, 0], scores_2d[triangle_emph, 1],
            s=105, c=point_colors[triangle_emph], marker="^", edgecolors="black",
            linewidths=1.2, zorder=4
        )

    # labels for emphasized IDs only (retain the previous behavior)
    dx = 0.01 * (scores_2d[:, 0].max() - scores_2d[:, 0].min() + 1e-9)
    dy = 0.01 * (scores_2d[:, 1].max() - scores_2d[:, 1].min() + 1e-9)
    for i, sid in enumerate(sample_ids):
        if is_emph[i]:
            ax.text(
                scores_2d[i, 0] + dx,
                scores_2d[i, 1] + dy,
                sid,
                fontsize=9,
                fontweight="bold",
                color="black",
                zorder=5
            )

    # feature vectors
    x_span = (scores_2d[:, 0].max() - scores_2d[:, 0].min()) if scores_2d.shape[0] > 1 else 1.0
    y_span = (scores_2d[:, 1].max() - scores_2d[:, 1].min()) if scores_2d.shape[0] > 1 else 1.0
    base_scale = 0.35 * min(x_span, y_span)
    scale = base_scale * vector_length_multiplier

    for j in range(len(feature_names)):
        label_clean = str(feature_names[j]).replace("_MolePct", "")
        ax.arrow(
            0, 0,
            loadings_2d[j, 0] * scale,
            loadings_2d[j, 1] * scale,
            head_width=0.02 * scale,
            length_includes_head=True,
            color="red"
        )
        ax.text(
            loadings_2d[j, 0] * scale * 1.08,
            loadings_2d[j, 1] * scale * 1.08,
            label_clean,
            fontsize=12,
            color="red"
        )

    # Legend: show the near-BGC triangle in addition to the display-group colors (circles)
    legend_order = sorted(
        [c for c in pd.unique(point_groups) if c != "Unassigned"],
        key=lambda x: int(re.sub(r"[^0-9]", "", str(x))) if re.search(r"\d+", str(x)) else 999
    )
    handles = [
        Line2D([0], [0], marker="o", linestyle="", markersize=6,
               markerfacecolor=group_to_color.get(group, default_point_color),
               markeredgecolor="black", markeredgewidth=0.3, label=group)
        for group in legend_order
    ]
    if np.any(point_groups == "Unassigned"):
        handles.append(
            Line2D([0], [0], marker="o", linestyle="", markersize=6,
                   markerfacecolor=default_point_color, markeredgecolor="black",
                   markeredgewidth=0.3, label="Unassigned")
        )
    handles.append(
        Line2D([0], [0], marker="^", linestyle="", markersize=8,
               markerfacecolor="white", markeredgecolor="black",
               markeredgewidth=0.8, label="Near BGC (triangle)")
    )
    ax.legend(handles=handles, title="Display groups / marker", bbox_to_anchor=(1.02, 1),
              loc="upper left", frameon=False)

    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel("PC1", fontsize=12)
    ax.set_ylabel("PC2", fontsize=12)
    # Figure title is intentionally omitted for manuscript-ready output.
    fig.tight_layout()

    fig.savefig(f"{out_prefix}_biplot.png", dpi=dpi, bbox_inches="tight")
    fig.savefig(f"{out_prefix}_biplot.pdf", bbox_inches="tight")
    plt.close(fig)

    matched_near_bgc_base_ids = sorted({
        _base_id(sid) for sid, flag in zip(sample_ids, is_near_bgc) if flag
    })
    print(f"Near BGC IDs plotted as triangles: {len(matched_near_bgc_base_ids)} / {len(set(map(_base_id, near_bgc_ids)))}")
    print(f"Saved biplot: {out_prefix}_biplot.png/.pdf")


In [ ]:
# -----------------------------
# PCA outputs (scores / explained variance / loadings)
# -----------------------------
def export_pca_outputs(out_prefix, clean_ids, scores, pca, feature_names, dpi=300):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # scores
    pc_cols = [f"PC{i+1}" for i in range(scores.shape[1])]
    score_df = pd.DataFrame(scores, columns=pc_cols)
    score_df.insert(0, "ID", list(clean_ids))
    score_df.to_csv(f"{out_prefix}_pca_scores.csv", index=False)

    # explained variance (ratio + cumulative)
    evr = pca.explained_variance_ratio_
    cum = np.cumsum(evr)
    contrib_df = pd.DataFrame({
        "PC": [f"PC{i+1}" for i in range(len(evr))],
        "ExplainedVarianceRatio": evr,
        "CumulativeExplainedVarianceRatio": cum,
        "ExplainedVariance": pca.explained_variance_
    })
    contrib_df.to_csv(f"{out_prefix}_pca_variance.csv", index=False)

    # loadings (scaled by sqrt eigenvalue) for reporting
    loadings_scaled = pca.components_.T * np.sqrt(pca.explained_variance_)
    load_df = pd.DataFrame(loadings_scaled, index=list(feature_names), columns=pc_cols)
    load_df.to_csv(f"{out_prefix}_pca_loadings_scaled.csv")

    # scree plot (png/pdf)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(range(1, len(evr)+1), evr, marker="o")
    ax.set_xlabel("Principal Component")
    ax.set_ylabel("Explained Variance Ratio")
    ax.set_title(f"Scree plot: {out_prefix}")
    fig.tight_layout()
    fig.savefig(f"{out_prefix}_scree_plot.png", dpi=dpi)
    fig.savefig(f"{out_prefix}_scree_plot.pdf")
    plt.close(fig)

    print("Saved PCA outputs:")
    print(f"  - {out_prefix}_pca_scores.csv")
    print(f"  - {out_prefix}_pca_variance.csv")
    print(f"  - {out_prefix}_pca_loadings_scaled.csv")
    print(f"  - {out_prefix}_scree_plot.png/.pdf")

In [ ]:
# -----------------------------
# aaComp (MolePct) pipeline: ID clean -> MolePct -> corr drop -> StandardScaler PCA -> outputs
# -----------------------------
def run_pipeline(
    csv_path,
    id_mode,
    out_prefix,
    corr_threshold=0.60,
    n_components=10,
    label_ids=None,
    near_bgc_ids=None,
    make_biplot=True,
    vector_length_multiplier=2.0,
    itol_colorstrip_path=None
):
    import pandas as pd
    import re
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA

    print(f"\n=== run_pipeline (aaComp / StandardScaler PCA; no CLR): {out_prefix} ===")
    print("Input:", csv_path, "| id_mode:", id_mode)

    df = pd.read_csv(csv_path)

    id_col = df.columns[0]
    clean_ids = clean_id_series(df[id_col], mode=id_mode)

    id_to_color = {}
    id_to_group = {}
    group_to_color = DEFAULT_GROUP_COLORS.copy()
    if itol_colorstrip_path is not None:
        id_to_color, id_to_group, group_to_color = load_itol_colorstrip(itol_colorstrip_path)
        print("Loaded iTOL colorstrip:", itol_colorstrip_path)

    # MolePct features
    feature_cols = [c for c in df.columns if str(c).endswith("_MolePct")]
    if len(feature_cols) == 0:
        raise ValueError("No '*_MolePct' columns found. This pipeline is for aaComp MolePct files.")

    X_raw = df[feature_cols].copy().astype(float)
    X_raw.index = clean_ids.tolist()

    # QC: these physicochemical property categories are overlapping and are not
    # expected to sum to 100%. This check is exported to document that CLR is not used.
    qc_df = pd.DataFrame({
        "ID": X_raw.index,
        "MolePctRowSum": X_raw.sum(axis=1).values,
        "NZeroFeatures": (X_raw == 0).sum(axis=1).values,
        "NMissingFeatures": X_raw.isna().sum(axis=1).values,
    })
    qc_df.to_csv(f"{out_prefix}_molepct_qc_noCLR.csv", index=False)
    print(
        "MolePct row-sum range (overlapping categories; not normalized): "
        f"{X_raw.sum(axis=1).min():.3f}–{X_raw.sum(axis=1).max():.3f}"
    )

    if X_raw.isna().any().any():
        missing_cols = X_raw.columns[X_raw.isna().any()].tolist()
        raise ValueError(f"Missing values found in MolePct columns: {missing_cols}")

    # Correlation filtering is performed on the original MolePct features, not CLR space.
    protected_features = [c for c in PROTECTED_MOLEPCT_FEATURES if c in X_raw.columns]
    print("Protected features (always retained if present):", protected_features)

    X_reduced, dropped_cols, corr_full, cluster_df = drop_highly_correlated_features(
        X_raw,
        threshold=corr_threshold,
        protected_features=protected_features
    )
    corr_reduced = X_reduced.corr()

    # export corr outputs (CSV + PNG/PDF heatmaps)
    export_corr_outputs(corr_full, corr_reduced, dropped_cols, out_prefix=out_prefix, dpi=300, cluster_df=cluster_df)

    print("Selected features after correlation filtering:", list(X_reduced.columns))
    print("Dropped features:", dropped_cols)

    # StandardScaler-only PCA.
    # This treats MolePct columns as overlapping physicochemical descriptors.
    scaler = StandardScaler()
    Xz = scaler.fit_transform(X_reduced.values)

    pca = PCA(n_components=min(n_components, Xz.shape[1]))
    scores = pca.fit_transform(Xz)

    evr = pca.explained_variance_ratio_
    pc1, pc2 = evr[0] * 100, evr[1] * 100

    # biplot uses components_ in the standardized feature space.
    loadings_2d = (
    pca.components_[:2, :].T* np.sqrt(pca.explained_variance_[:2]))

    if make_biplot:
        pca_biplot_emph_only(
            scores_2d=scores[:, :2],
            loadings_2d=loadings_2d,
            feature_names=X_reduced.columns.tolist(),
            sample_ids=X_reduced.index.tolist(),
            title=f"{out_prefix} PCA Biplot (PC1 {pc1:.1f}%, PC2 {pc2:.1f}%)\n"
                  f"aaComp StandardScaler; no CLR; CorrDrop |r|>{corr_threshold}",
            out_prefix=out_prefix,
            dpi=600,
            vector_length_multiplier=vector_length_multiplier,
            emph_ids=label_ids,
            near_bgc_ids=near_bgc_ids,
            id_to_color=id_to_color,
            id_to_group=id_to_group,
            group_to_color=group_to_color
        )

    # export PCA outputs
    export_pca_outputs(
        out_prefix=out_prefix,
        clean_ids=X_reduced.index.tolist(),
        scores=scores,
        pca=pca,
        feature_names=X_reduced.columns.tolist(),
        dpi=300
    )

    # Export the standardized matrix used for PCA for reproducibility.
    Xz_df = pd.DataFrame(Xz, index=X_reduced.index, columns=X_reduced.columns)
    Xz_df.insert(0, "ID", Xz_df.index)
    Xz_df.to_csv(f"{out_prefix}_standardized_features_for_PCA.csv", index=False)

    # For reference, also export the display-group assignment for each ID
    group_df = pd.DataFrame({
        "ID": X_reduced.index.tolist(),
        "Display_group": [id_to_group.get(i, id_to_group.get(re.sub(r"_\d+$", "", i), np.nan)) for i in X_reduced.index.tolist()],
        "Color": [id_to_color.get(i, id_to_color.get(re.sub(r"_\d+$", "", i), np.nan)) for i in X_reduced.index.tolist()],
    })
    group_df.to_csv(f"{out_prefix}_itol_display_group_assignment.csv", index=False)

    print("Saved:")
    print(f"  - {out_prefix}_molepct_qc_noCLR.csv")
    print(f"  - {out_prefix}_standardized_features_for_PCA.csv")
    print("=== Done ===")


# Notebook variant: support-prioritized display groups and AO090005001236 highlighting

This version processes the Domain, FL, and non_DBD amino-acid-composition MolePct tables by StandardScaler PCA without CLR. Each PCA biplot uses the corresponding `*_itol_colorstrip_support_k10.txt` file. The colors denote support-prioritized **display groups (G1--G10)** and are not treated as named biological clades.

The 48 IDs in `Near_BGC_AoID_list` are plotted as triangles. `AO090005001236` is also included in `EMPH_IDS`; therefore it is plotted as a larger, black-edged triangle and labeled with its ID. Other emphasized IDs retain the previous styling.


In [ ]:
# -----------------------------
# Input-file settings (upload files to /content in Colab)
# -----------------------------
from pathlib import Path

BASE_DIR = Path("/content")

AA_PATH = BASE_DIR / "aaComp_MolePct.csv"
FL_PATH = BASE_DIR / "FL_aaComp_MolePct.csv"
NON_DBD_PATH = BASE_DIR / "non_DBD_aaComp_MolePct.csv"

DOMAIN_COLORSTRIP_PATH = BASE_DIR / "Domain_itol_colorstrip_support_k10.txt"
FL_COLORSTRIP_PATH = BASE_DIR / "FL_itol_colorstrip_support_k10.txt"
NON_DBD_COLORSTRIP_PATH = BASE_DIR / "non_DBD_itol_colorstrip_support_k10.txt"

# Use a separate output directory for each analysis
OUTPUT_ROOT = BASE_DIR / "PCA_outputs"
OUTPUT_DIRS = {
    "DBD": OUTPUT_ROOT / "DBD",
    "FL": OUTPUT_ROOT / "FL",
    "non_DBD": OUTPUT_ROOT / "non_DBD",
}
for output_dir in OUTPUT_DIRS.values():
    output_dir.mkdir(parents=True, exist_ok=True)

print("Output root:", OUTPUT_ROOT)
for name, output_dir in OUTPUT_DIRS.items():
    print(f"  {name}: {output_dir}")


In [ ]:
# -----------------------------
# Run: process the three CSV files (DBD / FL / non_DBD; G1-G10 display groups)
# -----------------------------
run_pipeline(
    csv_path=str(AA_PATH),
    id_mode="SM00066",
    out_prefix=str(OUTPUT_DIRS["DBD"] / "DBD_aaComp_StdPCA_noCLR_unprotected"),
    corr_threshold=0.60,
    n_components=10,
    label_ids=EMPH_IDS,
    near_bgc_ids=NEAR_BGC_IDS,
    itol_colorstrip_path=str(DOMAIN_COLORSTRIP_PATH)
)

run_pipeline(
    csv_path=str(FL_PATH),
    id_mode="FL",
    out_prefix=str(OUTPUT_DIRS["FL"] / "FL_aaComp_StdPCA_noCLR_unprotected"),
    corr_threshold=0.60,
    n_components=10,
    label_ids=EMPH_IDS,
    near_bgc_ids=NEAR_BGC_IDS,
    itol_colorstrip_path=str(FL_COLORSTRIP_PATH)
)

run_pipeline(
    csv_path=str(NON_DBD_PATH),
    id_mode="non_DBD",
    out_prefix=str(OUTPUT_DIRS["non_DBD"] / "non_DBD_aaComp_StdPCA_noCLR_unprotected"),
    corr_threshold=0.60,
    n_components=10,
    label_ids=EMPH_IDS,
    near_bgc_ids=NEAR_BGC_IDS,
    itol_colorstrip_path=str(NON_DBD_COLORSTRIP_PATH)
)


## Organize output files and download them together

Separate the analysis results into `DBD`, `FL`, and `non_DBD` directories. The following cell creates individual ZIP archives for each directory and a combined ZIP containing all three. In Colab, only the combined ZIP is downloaded automatically.


In [ ]:
# -----------------------------
# Create individual ZIP archives for DBD / FL / non_DBD and download all results together
# -----------------------------
import shutil
from pathlib import Path

# Individual ZIP archives for each directory (available separately from the Colab Files pane if needed)
individual_archives = {}
for analysis_name, output_dir in OUTPUT_DIRS.items():
    archive_base = BASE_DIR / f"PCA_outputs_{analysis_name}"
    archive_path = Path(shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(OUTPUT_ROOT),
        base_dir=analysis_name,
    ))
    individual_archives[analysis_name] = archive_path
    print(f"Created {analysis_name} ZIP: {archive_path}")

# Combined ZIP preserving the DBD, FL, and non_DBD directories
combined_base = BASE_DIR / "PCA_outputs_DBD_FL_non_DBD"
combined_zip = Path(shutil.make_archive(
    str(combined_base),
    "zip",
    root_dir=str(OUTPUT_ROOT),
))
print("Created combined ZIP:", combined_zip)

# Inspect ZIP contents
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(OUTPUT_ROOT))

# Download the combined ZIP once in Google Colab
try:
    from google.colab import files
    files.download(str(combined_zip))
except ImportError:
    print("Executed outside Google Colab. Retrieve the ZIP manually:", combined_zip)
